# Validação 28 — indicadores separados

## Objetivo

Comprovar que a aplicação não resume a análise em uma porcentagem ambígua. O resultado deve separar: cobertura da busca, compatibilidade das evidências e confiança metodológica.

## Configuração

O cenário é controlado e serve para validar regras de apresentação. Os três indicadores não compartilham uma escala numérica; por isso, uma tabela é mais adequada que um gráfico, que poderia sugerir comparabilidade inexistente.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

from fatofake.verification_cards import build_verification_indicators

## Cenário controlado

Cinco documentos foram recuperados. Quatro têm trechos analisados: dois compatíveis, um divergente e um apenas contextual. Dois foram lidos em texto completo. Os níveis metodológicos são heterogêneos.

In [2]:
def article(relation=None, *, access='ABSTRACT_ONLY', quality='UNKNOWN', citations=0, retracted=False):
    return {
        'access_level': access,
        'assessments': [] if relation is None else [{'relation': relation}],
        'quality': {
            'level': quality, 'is_retracted': retracted,
            'trial_registrations': [], 'datasets': [],
        },
        'retrieval': {'citation_count': citations, 'related_work_count': citations},
    }

articles = [
    article('SUPPORTS', access='FULL_TEXT', quality='HIGH', citations=1200),
    article('SUPPORTS', access='FULL_TEXT', quality='MODERATE', citations=3),
    article('CONTRADICTS', quality='LOW', citations=800),
    article('NEUTRAL', quality='UNKNOWN', citations=50),
    article(None, quality='UNKNOWN', citations=4000),
]
indicators = build_verification_indicators(articles=articles, research_context='BASIC_SCIENCE')
indicators

{'search_coverage': {'status': 'PARTIAL',
  'label': 'Cobertura parcial dos documentos recuperados',
  'retrieved_count': 5,
  'assessed_count': 4,
  'directly_comparable_count': 3,
  'full_text_count': 2,
  'abstract_only_count': 2,
  'explanation': '4 de 5 documentos recuperados foram analisados; 2 com texto completo e 3 responderam diretamente à alegação.'},
 'evidence_compatibility': {'status': 'DIVERGENT',
  'label': 'Evidências divergentes',
  'supporting_count': 2,
  'contradicting_count': 1,
  'mixed_count': 0,
  'context_only_count': 1,
  'inconclusive_count': 0,
  'compared_article_count': 4,
  'explanation': 'Entre 4 artigos analisados: 2 compatíveis, 1 divergentes, 0 mistos, 1 apenas contextuais e 0 inconclusivos.'},
 'methodological_confidence': {'level': 'MIXED',
  'label': 'Confiança metodológica heterogênea',
  'assessed_article_count': 4,
  'quality_counts': {'HIGH': 1, 'MODERATE': 1, 'LOW': 1, 'UNKNOWN': 1},
  'study_design_counts': {'UNKNOWN': 4},
  'retracted_count'

## Resultados

In [3]:
coverage = indicators['search_coverage']
compatibility = indicators['evidence_compatibility']
methodology = indicators['methodological_confidence']

[
    {
        'indicador': 'Cobertura da busca',
        'classificação': coverage['label'],
        'base observada': f"{coverage['assessed_count']} de {coverage['retrieved_count']} analisados; {coverage['full_text_count']} textos completos",
    },
    {
        'indicador': 'Compatibilidade das evidências',
        'classificação': compatibility['label'],
        'base observada': f"{compatibility['supporting_count']} compatíveis; {compatibility['contradicting_count']} divergentes; {compatibility['context_only_count']} contextuais",
    },
    {
        'indicador': 'Confiança metodológica',
        'classificação': methodology['label'],
        'base observada': str(methodology['quality_counts']),
    },
]

[{'indicador': 'Cobertura da busca',
  'classificação': 'Cobertura parcial dos documentos recuperados',
  'base observada': '4 de 5 analisados; 2 textos completos'},
 {'indicador': 'Compatibilidade das evidências',
  'classificação': 'Evidências divergentes',
  'base observada': '2 compatíveis; 1 divergentes; 1 contextuais'},
 {'indicador': 'Confiança metodológica',
  'classificação': 'Confiança metodológica heterogênea',
  'base observada': "{'HIGH': 1, 'MODERATE': 1, 'LOW': 1, 'UNKNOWN': 1}"}]

## Verificações de segurança

A primeira checagem impede o retorno de um `score` geral. A segunda demonstra que aumentar citações e ramificações não modifica a confiança metodológica. A terceira garante que uma retratação seja tratada como alerta grave.

In [4]:
assert set(indicators) == {'search_coverage', 'evidence_compatibility', 'methodological_confidence'}
assert 'score' not in str(indicators).casefold()
assert coverage['assessed_count'] == 4 and coverage['retrieved_count'] == 5
assert compatibility['supporting_count'] == 2
assert compatibility['contradicting_count'] == 1
assert compatibility['context_only_count'] == 1
assert methodology['level'] == 'MIXED'

few_citations = build_verification_indicators(
    articles=[article('SUPPORTS', quality='MODERATE', citations=0)],
    research_context='BASIC_SCIENCE',
)['methodological_confidence']['level']
many_citations = build_verification_indicators(
    articles=[article('SUPPORTS', quality='MODERATE', citations=100000)],
    research_context='BASIC_SCIENCE',
)['methodological_confidence']['level']
assert few_citations == many_citations == 'MODERATE'

retracted = build_verification_indicators(
    articles=[article('SUPPORTS', quality='HIGH', retracted=True)],
    research_context='BASIC_SCIENCE',
)['methodological_confidence']
assert retracted['level'] == 'CRITICAL_ALERT'
print('OK: três indicadores independentes, sem percentual geral e sem influência de citações na qualidade.')

OK: três indicadores independentes, sem percentual geral e sem influência de citações na qualidade.


## Próximos passos

A etapa valida a separação conceitual e o contrato dos indicadores; não valida a qualidade científica dos artigos. A próxima etapa priorizará metadados determinísticos para classificar o desenho do estudo e ampliará a ficha do artigo enviado.